# Evaluation: Error Related To Cosine Target Period

This notebook keeps the model fixed (`Style4-Past36-LSTM`) and changes the cosine target period. It asks how prediction error changes when the target moves faster or slower.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd()
if REPO.name == "notebook":
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.util import (
    CAMERA_MAX,
    CANONICAL_PERIOD,
    COLORS,
    EVAL_STEPS,
    FREQ_PERIODS_STEPS,
    FREQ_SWEEP_EVAL_STEPS,
    STEP_MIN,
    experiment_runtime,
    load_eval_pool,
    make_cosine_target,
    plot_error_time,
    plot_single_rollout_trajectory,
    plot_trajectory_group,
    rising_mask,
    run_ode_mpc,
    show_figure,
)


## Rising/Falling Error Across Periods

In [2]:

def compute_freq_sweep(experiment_name, eval_style=4, periods=FREQ_PERIODS_STEPS, eval_steps=FREQ_SWEEP_EVAL_STEPS):
    cfg, model, ode, horizon, past_steps, camera_max = experiment_runtime(experiment_name)
    rows = []
    for period in periods:
        desired = make_cosine_target(period, eval_steps, camera_max)
        rise_mask = rising_mask(desired)
        fall_mask = ~rise_mask
        all_nmae = []
        all_rise = []
        all_fall = []
        for record in load_eval_pool(style=eval_style):
            ode_response, _ = run_ode_mpc(
                model,
                record.observed[:past_steps],
                record.stimulus[:past_steps],
                desired,
                record.cell_E,
                horizon,
                ode,
                camera_max,
                context_steps=36,
            )
            err = np.abs(ode_response - desired) / camera_max
            all_nmae.append(float(np.mean(err)))
            all_rise.append(float(np.mean(err[rise_mask])))
            all_fall.append(float(np.mean(err[fall_mask])))
        rows.append({
            "period_steps": period,
            "period_hours": round(period * STEP_MIN / 60, 2),
            "nmae_mean": float(np.mean(all_nmae)),
            "nmae_std": float(np.std(all_nmae)),
            "rising_nmae_mean": float(np.mean(all_rise)),
            "falling_nmae_mean": float(np.mean(all_fall)),
        })
    return {"freq_sweep": rows}


def compute_freq_trajectories(experiment_name="Style4-Past36-LSTM", eval_style=4, periods=FREQ_PERIODS_STEPS, eval_steps=FREQ_SWEEP_EVAL_STEPS):
    cfg, model, ode, horizon, past_steps, camera_max = experiment_runtime(experiment_name)
    record = load_eval_pool(style=eval_style, n=1)[0]
    warmup_obs = record.observed[:past_steps]
    warmup_stimulus = record.stimulus[:past_steps]
    freq_trajs = []
    for period in periods:
        target = make_cosine_target(period, eval_steps, camera_max)
        ode_response, applied_stimulus = run_ode_mpc(
            model,
            warmup_obs,
            warmup_stimulus,
            target,
            record.cell_E,
            horizon,
            ode,
            camera_max,
            context_steps=past_steps,
        )
        freq_trajs.append({
            "period_steps": period,
            "period_hours": round(period * STEP_MIN / 60, 2),
            "times_eval_hr": [round(i * STEP_MIN / 60, 4) for i in range(len(ode_response))],
            "ode_resp_au": ode_response.tolist(),
            "target_au": target.tolist(),
        })
    return {"freq_trajs": freq_trajs}


def plot_frequency_error(title, save_name=None):
    rows = compute_freq_sweep("Style4-Past36-LSTM")["freq_sweep"]
    periods = [row["period_hours"] for row in rows]
    overall = [row["nmae_mean"] for row in rows]
    rising = [row["rising_nmae_mean"] for row in rows]
    falling = [row["falling_nmae_mean"] for row in rows]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(periods, overall, "o-", color="gray", lw=2, ms=7, label="Overall nMAE", zorder=2)
    ax.plot(periods, rising, "^--", color=COLORS[2], lw=2, ms=8, label="Rising phase", zorder=3)
    ax.plot(periods, falling, "v--", color=COLORS[3], lw=2, ms=8, label="Falling phase", zorder=3)
    ax.fill_between(periods, rising, falling, alpha=0.12, color=COLORS[3])
    for period, rise, fall in zip(periods, rising, falling):
        ratio = fall / rise if rise > 0 else 0
        ax.annotate(f"x{ratio:.1f}", (period, (rise + fall) / 2), textcoords="offset points", xytext=(8, 0), fontsize=8)
    ax.axvline(8.0, color="navy", lw=1.2, ls=":", alpha=0.6, label="Canonical 8h")
    ax.set_xlabel("Target cosine period (hr)")
    ax.set_ylabel("nMAE")
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(periods)
    ax.legend(fontsize=9)
    plt.tight_layout()
    show_figure(title, save_name)


def plot_frequency_trajectories(title, save_name=None):
    rows = compute_freq_trajectories("Style4-Past36-LSTM")["freq_trajs"]
    fig, ax = plt.subplots(figsize=(8, 5))
    for row, color in zip(rows, COLORS):
        period_h = row["period_hours"]
        times = np.array(row["times_eval_hr"])
        target = np.array(row["target_au"])
        response = np.array(row["ode_resp_au"])
        mask = times <= period_h * 1.5
        ax.plot(times[mask], target[mask], "--", color=color, lw=1, alpha=0.5)
        ax.plot(times[mask], response[mask], "-", color=color, lw=1.8, label=f"{period_h:.0f}h period")
    ax.set_xlabel("Time (hr)")
    ax.set_ylabel("GFP fluorescence (AU)")
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0, CAMERA_MAX * 0.7)
    ax.legend(fontsize=8)
    plt.tight_layout()
    show_figure(title, save_name)

plot_frequency_error("Error vs cosine target period", save_name="evaluation-ErrorRelateToCosineTargetPeriod-error")


saved figure: [asset/evaluation-ErrorRelateToCosineTargetPeriod-error.png](../asset/evaluation-ErrorRelateToCosineTargetPeriod-error.png)

## Trajectory Examples Across Periods

In [3]:
plot_frequency_trajectories("Trajectory examples across cosine target periods", save_name="evaluation-ErrorRelateToCosineTargetPeriod-trajectory")


saved figure: [asset/evaluation-ErrorRelateToCosineTargetPeriod-trajectory.png](../asset/evaluation-ErrorRelateToCosineTargetPeriod-trajectory.png)